# ZoneGuard: resume YOLO11n on a free Colab T4

1. Select **Runtime ? Change runtime type ? T4 GPU**. No paid accelerator is required.
2. On your machine, clone the repository and run `python scripts/package_colab.py`.
3. Upload `dist/zoneguard-colab.zip` below. Git credentials are excluded.
4. The archived checkpoint contains five completed epochs. Data must match its original SHA-256 before optimization resumes.

Free Colab can disconnect. Download checkpoint/results periodically. Keep the final test split out of parameter tuning; two parking-video events are a regression check, not a field-accuracy benchmark.


In [ ]:
from google.colab import files
from pathlib import Path
import zipfile, json, hashlib
uploaded = files.upload()
assert len(uploaded)==1, "Upload one ZoneGuard archive"
name, data = next(iter(uploaded.items()))
archive=Path('/content/zoneguard-upload.zip');archive.write_bytes(data)
root=Path('/content/zoneguard');root.mkdir(exist_ok=True)
with zipfile.ZipFile(archive) as z:
    assert all((root/name).resolve().is_relative_to(root.resolve()) for name in z.namelist())
    z.extractall(root)
metadata=json.loads((root/'COLAB_PACKAGE.json').read_text())
for name,digest in metadata['files_sha256'].items():
    assert hashlib.file_digest((root/name).open('rb'),'sha256').hexdigest()==digest,name
print('Verified package', metadata['source_commit'], 'dirty:', metadata['working_tree_dirty'])


In [ ]:
import subprocess,sys,shutil,os
assert shutil.which('nvidia-smi'), 'Select a GPU runtime first'
pidfile=root/'runtime/colab-training.pid'
if pidfile.exists():
    import psutil
    assert not psutil.pid_exists(int(pidfile.read_text())), 'A worker is already running'
(root/'runtime').mkdir(exist_ok=True)
with (root/'runtime/colab-training.log').open('w') as log:
    worker=subprocess.Popen([sys.executable,'-u',str(root/'scripts/colab_bootstrap.py')],cwd=root,stdout=log,stderr=subprocess.STDOUT,start_new_session=True)
pidfile.write_text(str(worker.pid))
print('Background worker:',worker.pid)


## Current status
Run this cell again to refresh. Its output is a snapshot, not an automatic monitor.


In [ ]:
for name in ['reports/colab-bootstrap.json','reports/training-live.json','reports/colab-checkpoint-selection.json']:
    p=root/name
    if p.exists(): print(name,p.read_text())
log=root/'runtime/colab-training.log'
if log.exists():
    with log.open('rb') as f:
        f.seek(max(0,log.stat().st_size-3000));print(f.read().decode(errors='replace'))


## Download a recovery/results bundle
Run after an epoch has been saved, and again when evaluation finishes. The bundle excludes raw datasets and account credentials. Extract it into the same checkout to recover checkpoints and reports.


In [ ]:
import gzip
bundle=Path('/content/zoneguard-results.zip')
paths=list((root/'reports').glob('*.json'))+list((root/'reports').glob('*.csv'))
paths+=list((root/'runs/zoneguard_compact/weights').glob('*.pt'))
paths+=list((root/'runs/zoneguard_compact').glob('results.csv'))
if (root/'models/vehicle-large.pt').exists():paths.append(root/'models/vehicle-large.pt')
with zipfile.ZipFile(bundle,'w',zipfile.ZIP_DEFLATED,compresslevel=1) as z:
    for p in paths:z.write(p,p.relative_to(root).as_posix())
    manifest=root/'datasets/compact/manifest.json'
    if manifest.exists():z.writestr('data/compact-manifest.json.gz',gzip.compress(manifest.read_bytes(),mtime=0))
files.download(str(bundle))
